<div dir="rtl">

# الترم ٢ · الوحدة ١: يعني إيه تعلّم الآلة؟

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdelrahman-abozarifa04/ai-for-kids-course/blob/main/notebooks/teens/term-2/04-what-is-ml.ipynb)

في مشروع الترم ١ إحنا اللي **كتبنا القواعد**. في تعلّم الآلة (Machine Learning) **الكمبيوتر هو اللي بيطلّع القواعد** من الأمثلة.

```
البرمجة العادية:   قواعد + بيانات  ←  إجابات
تعلّم الآلة:        بيانات + إجابات ←  قواعد (model)
```

</div>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay
from sklearn.cluster import KMeans

sns.set_theme(style="whitegrid")

<div dir="rtl">

## ١. البيانات: زهور الـ Iris

١٥٠ زهرة من ٣ أنواع. لكل زهرة ٤ قياسات (features) بالسنتيمتر، والنوع هو الإجابة (label).

</div>

In [ ]:
iris = load_iris(as_frame=True)
df = iris.frame.copy()
df["species"] = df["target"].map(dict(enumerate(iris.target_names)))
df.sample(5, random_state=1)

In [ ]:
sns.pairplot(df.drop(columns="target"), hue="species", height=1.8)
plt.show()

<div dir="rtl">

**لاحظ:** نوع `setosa` منفصل خالص عن الباقي. الكمبيوتر هيلاحظ ده برضو.

</div>

<div dir="rtl">

## ٢. Supervised Learning: تعلّم بإشراف (عندنا الإجابات)

### الخطوة أ: نقسّم البيانات تدريب واختبار
زي الزوزو في مسار الأطفال: بنعلّم على جزء، ونختبر على جزء **ما شافهوش قبل كده**.

</div>

In [ ]:
X = df[iris.feature_names]
y = df["species"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print("Train:", X_train.shape, "Test:", X_test.shape)

<div dir="rtl">

### الخطوة ب: ندرّب الموديل

**K-Nearest Neighbors (KNN):** علشان يعرف نوع زهرة جديدة، بيبص على أقرب **k** زهور ليها في التدريب وياخد رأي الأغلبية.

</div>

In [ ]:
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train, y_train)          # التعلم!

<div dir="rtl">

### الخطوة ج: نختبر

</div>

In [ ]:
pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, pred):.0%}")
ConfusionMatrixDisplay.from_predictions(y_test, pred, cmap="Blues")
plt.title("Confusion matrix")
plt.show()

<div dir="rtl">

**الـ Confusion Matrix:** القطر = إجابات صح. أي رقم بره القطر = الموديل لخبط بين نوعين.

</div>

<div dir="rtl">

### الخطوة د: نستخدم الموديل على زهرة جديدة

</div>

In [ ]:
new_flower = pd.DataFrame([[5.0, 3.4, 1.5, 0.2]], columns=iris.feature_names)
print("Prediction:", model.predict(new_flower)[0])

<div dir="rtl">

## ٣. "التعلم بيتحسن مع البيانات"

هندرّب نفس الموديل على كميات بيانات مختلفة ونشوف الدقة.

</div>

In [ ]:
sizes, scores = [3, 6, 10, 20, 40, 80, 112], []
rng = np.random.default_rng(0)
for size in sizes:
    accs = []
    for _ in range(30):
        idx = rng.choice(len(X_train), size=size, replace=False)
        k = min(3, size)
        m = KNeighborsClassifier(n_neighbors=k).fit(X_train.iloc[idx], y_train.iloc[idx])
        accs.append(m.score(X_test, y_test))
    scores.append(np.mean(accs))

plt.figure(figsize=(7, 4))
plt.plot(sizes, scores, marker="o", color="#4F46E5")
plt.xlabel("Number of training examples")
plt.ylabel("Test accuracy")
plt.title("More data -> better model")
plt.show()

<div dir="rtl">

## ٤. Unsupervised Learning: تعلّم من غير إشراف (مفيش إجابات)

لو شلنا عمود النوع، هل الكمبيوتر يقدر يكتشف إن فيه مجموعات؟ **K-Means** بيحاول يقسّم البيانات لـ k مجموعات متشابهة.

</div>

In [ ]:
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
df["cluster"] = kmeans.fit_predict(X)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.scatterplot(data=df, x="petal length (cm)", y="petal width (cm)", hue="species", ax=axes[0])
axes[0].set_title("Real species (hidden from K-Means)")
sns.scatterplot(data=df, x="petal length (cm)", y="petal width (cm)", hue="cluster", palette="Set2", ax=axes[1])
axes[1].set_title("Groups found by K-Means")
plt.show()

pd.crosstab(df["species"], df["cluster"])

<div dir="rtl">

**النتيجة:** K-Means لقى المجموعات تقريباً لوحده، من غير ما حد يقوله الأسماء!

</div>

<div dir="rtl">

## 🏋️ تمرين ١: جرّب قيم k مختلفة
درّب KNN بـ `n_neighbors` = 1, 3, 7, 15, 50 واطبع الدقة لكل واحد. إيه اللي بيحصل لما k تكبر أوي؟

<details><summary>الحل</summary>

```python
for k in [1, 3, 7, 15, 50]:
    acc = KNeighborsClassifier(n_neighbors=k).fit(X_train, y_train).score(X_test, y_test)
    print(f"k={k:>2}  accuracy={acc:.0%}")
# لما k تكبر أوي، الموديل بيبص على زهور بعيدة كتير وبيبدأ يغلط.
```
</details>

</div>

In [ ]:
# اكتب الحل هنا

<div dir="rtl">

## 🏋️ تمرين ٢: موديل بـ feature واحدة
درّب KNN باستخدام `sepal width (cm)` بس. الدقة هتبقى كام؟ وليه؟

<details><summary>الحل</summary>

```python
cols = ["sepal width (cm)"]
acc = KNeighborsClassifier(n_neighbors=5).fit(X_train[cols], y_train).score(X_test[cols], y_test)
print(f"Accuracy with one weak feature: {acc:.0%}")
# الدقة بتقع، لأن عرض السبلة لوحده مش بيفرّق بين الأنواع كويس.
```
</details>

</div>

In [ ]:
# اكتب الحل هنا

<div dir="rtl">

## ✅ لخّص
| المصطلح | المعنى |
|---|---|
| Feature | خاصية (قياس) بنديها للموديل |
| Label | الإجابة الصح |
| Training set | الأمثلة اللي الموديل بيتعلم منها |
| Test set | أمثلة جديدة علشان نقيس الموديل بجد |
| Supervised | عندنا الإجابات (تصنيف، توقع أرقام) |
| Unsupervised | مفيش إجابات، الموديل يلاقي مجموعات لوحده |

</div>